# Demo 02a — Gradient-Based Optimization

**Class 02 · Block 2** (slides: `slides/02_optimization.md`)

Six live demos:

* **G1 — Gradient descent on a convex function:** the derivative by hand, by finite differences, and with JAX.
* **G2 — The learning rate:** too small, just right, oscillating, diverging.
* **G3 — A rugged (noisy-looking) function:** local minima and the starting point.
* **G4 — Newton's method:** curvature, quadratic convergence, and how it fails.
* **G5 — Momentum, RMSProp and Adam:** ill-conditioned valleys and mini-batch noise.
* **G6 — JAX for custom models and losses:** a damped oscillator with a Huber loss, and differentiating through an ODE solver.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)  # float64: clean convergence plots
plt.rcParams["figure.figsize"] = (9, 4)
print("JAX", jax.__version__, "on", jax.devices())

## The two 1D test functions

The same two functions are used in the blind-optimization notebook, so both families can be compared.

$$f_{\text{convex}}(x) = (x-2)^2 + 1 \qquad\qquad f_{\text{rugged}}(x) = \frac{x^2}{5} + \sin(3x) + 0.3\,\sin(11x)$$

The rugged one is a smooth bowl (the *signal*) plus fast ripples that look like *noise*.
Its global minimum is at $x^\star \approx -0.67$, with several local minima around it.

In [ ]:
def f_convex(x):
    return (x - 2.0) ** 2 + 1.0


def f_rugged(x):
    return x**2 / 5 + jnp.sin(3 * x) + 0.3 * jnp.sin(11 * x)


grid = jnp.linspace(-4, 4, 2001)
fig, axes = plt.subplots(1, 2)
for ax, f, name in zip(axes, [f_convex, f_rugged], ["convex", "rugged"]):
    ax.plot(grid, f(grid), color="tab:blue")
    i = int(jnp.argmin(f(grid)))
    ax.plot(grid[i], f(grid)[i], "r*", ms=14, label=f"global min x*={float(grid[i]):.2f}")
    ax.set(title=f"f_{name}(x)", xlabel="x")
    ax.legend()
plt.tight_layout()
plt.show()

## G1 — Gradient descent on a convex function

**By hand.** $f'(x) = 2(x-2)$. Setting $f'(x)=0$ gives $x^\star = 2$ directly: for this function we do not need an
iterative method at all. Gradient descent is for when we *cannot* solve $f'(x)=0$ in closed form, but it is easiest
to understand where we know the answer.

$$x_{k+1} = x_k - \eta\, f'(x_k) = x_k - 2\eta\,(x_k - 2)
\quad\Longrightarrow\quad x_{k+1} - 2 = (1 - 2\eta)\,(x_k - 2)$$

The error shrinks by the factor $|1-2\eta|$ at every step.

**Three ways to get the derivative:** symbolic (by hand), numerical (finite differences), automatic (JAX).

In [ ]:
def df_convex_manual(x):
    return 2.0 * (x - 2.0)


def finite_difference(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)


df_convex_jax = jax.grad(f_convex)

for x in [-2.0, 0.0, 3.5]:
    print(f"x={x:5.1f}  manual={df_convex_manual(x):8.4f}  "
          f"finite diff={float(finite_difference(f_convex, x)):8.4f}  jax.grad={float(df_convex_jax(x)):8.4f}")

In [ ]:
def gradient_descent(grad, x0, lr, n_steps):
    """Plain gradient descent. Returns the whole path, x_0 ... x_n."""
    x = jnp.asarray(x0, dtype=float)
    path = [x]
    for _ in range(n_steps):
        x = x - lr * grad(x)
        path.append(x)
    return jnp.stack(path)


path = gradient_descent(df_convex_jax, x0=-2.0, lr=0.1, n_steps=25)
print(" k        x_k      f(x_k)   f'(x_k)")
for k in range(6):
    xk = path[k]
    print(f"{k:2d}  {float(xk):9.4f}  {float(f_convex(xk)):9.4f}  {float(df_convex_jax(xk)):8.4f}")

fig, ax = plt.subplots()
ax.plot(grid, f_convex(grid), color="tab:blue")
ax.plot(path, f_convex(path), "o-", color="tab:red", ms=4, label="GD path, lr = 0.1")
ax.set(xlim=(-3, 4), ylim=(0, 27), xlabel="x", ylabel="f(x)", title="Gradient descent walks downhill")
ax.legend()
plt.show()

**Takeaway:** each step moves against the slope, by an amount proportional to the slope.
Near the minimum the slope vanishes, so the steps shrink by themselves.

## G2 — The learning rate

From $x_{k+1} - 2 = (1-2\eta)(x_k-2)$:

| $\eta$ | factor $1-2\eta$ | behaviour |
|:-:|:-:|:--|
| 0.05 | 0.9 | slow, monotone |
| 0.45 | 0.1 | fast |
| 0.95 | −0.9 | oscillates around $x^\star$ |
| 1.05 | −1.1 | diverges |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lr, color in zip([0.05, 0.45, 0.95, 1.05], ["tab:blue", "tab:green", "tab:orange", "tab:red"]):
    p = gradient_descent(df_convex_jax, x0=-2.0, lr=lr, n_steps=30)
    axes[0].plot(p[:8], f_convex(p[:8]), "o-", color=color, ms=4, label=f"lr = {lr}")
    axes[1].semilogy(jnp.abs(p - 2.0), "o-", color=color, ms=3, label=f"lr = {lr}")
axes[0].plot(grid, f_convex(grid), color="k", lw=1, zorder=0)
axes[0].set(xlim=(-4, 8), ylim=(0, 40), xlabel="x", ylabel="f(x)", title="first 8 steps")
axes[1].set(xlabel="iteration k", ylabel="|x_k - x*|", title="error (log scale)")
axes[1].legend()
plt.tight_layout()
plt.show()

**Takeaway:** the learning rate is the most important hyperparameter of gradient descent.
The safe range depends on the **curvature** $f''(x)$: here $f''=2$ and GD is stable for $\eta < 2/f'' = 1$.

## G3 — A rugged function: local minima

$$f'_{\text{rugged}}(x) = \frac{2x}{5} + 3\cos(3x) + 3.3\cos(11x)$$

Writing this by hand is already error-prone; JAX gets it right for free. We check both, then start gradient descent
from several points.

In [ ]:
def df_rugged_manual(x):
    return 2 * x / 5 + 3 * jnp.cos(3 * x) + 3.3 * jnp.cos(11 * x)


df_rugged = jax.jit(jax.grad(f_rugged))
xs = jnp.linspace(-4, 4, 9)
print("max |manual - jax| over 9 points:", float(jnp.max(jnp.abs(df_rugged_manual(xs) - jax.vmap(df_rugged)(xs)))))

starts = [-3.5, -2.0, -0.4, 1.0, 3.0]
fig, ax = plt.subplots()
ax.plot(grid, f_rugged(grid), color="tab:blue", lw=1)
for x0, color in zip(starts, plt.cm.viridis(np.linspace(0, 0.9, len(starts)))):
    p = gradient_descent(df_rugged, x0=x0, lr=0.02, n_steps=200)
    ax.plot(p, f_rugged(p), ".-", color=color, ms=3)
    ax.plot(p[-1], f_rugged(p[-1]), "o", color=color, ms=9, label=f"x0={x0:+.1f} -> x={float(p[-1]):+.2f}, f={float(f_rugged(p[-1])):+.2f}")
ax.set(xlabel="x", ylabel="f(x)", title="Gradient descent stops at the nearest valley")
ax.legend(fontsize=8)
plt.show()

In [ ]:
x0s = jnp.linspace(-4, 4, 400)


@jax.jit
def gd_final(x0, lr=0.02, n_steps=300):
    """The same GD loop, compiled with lax.fori_loop: returns only x_n."""
    return jax.lax.fori_loop(0, n_steps, lambda _, x: x - lr * jax.grad(f_rugged)(x), x0)


finals = jax.vmap(gd_final)(x0s)
fig, ax = plt.subplots()
ax.scatter(x0s, finals, c=f_rugged(finals), cmap="viridis", s=8)
ax.axhline(-0.67, color="r", ls="--", label="global minimum x* = -0.67")
ax.set(xlabel="starting point x0", ylabel="final x after 300 steps", title="Basins of attraction")
ax.legend()
plt.show()
print(f"starts that reach the global basin: {float(jnp.mean(jnp.abs(finals + 0.67) < 0.05)):.0%}")

**Takeaway:** gradient descent is a **local** method: it finds *a* minimum, the one whose basin contains $x_0$.
On a rugged landscape most starting points end in a local minimum. Blind, population-based methods (next notebook)
attack exactly this weakness.

## G4 — Newton's method

Approximate $f$ around $x_k$ by its second-order Taylor polynomial and jump to the minimum of that parabola:

$$f(x_k + \delta) \approx f(x_k) + f'(x_k)\,\delta + \tfrac12 f''(x_k)\,\delta^2
\quad\Longrightarrow\quad x_{k+1} = x_k - \frac{f'(x_k)}{f''(x_k)}$$

In $n$ dimensions: $\theta_{k+1} = \theta_k - H^{-1}\nabla\mathcal{L}(\theta_k)$ with $H$ the Hessian matrix.

* On a quadratic, the Taylor model is exact: **one step**.
* Near a minimum, the number of correct digits roughly **doubles** each step (quadratic convergence).
* No learning rate, but it needs $f''$ (the Hessian: $n^2$ entries, $O(n^3)$ to solve).

In [ ]:
def newton(f, x0, n_steps):
    df, d2f = jax.grad(f), jax.grad(jax.grad(f))
    x = jnp.asarray(x0, dtype=float)
    path = [x]
    for _ in range(n_steps):
        x = x - df(x) / d2f(x)
        path.append(x)
    return jnp.stack(path)


print("convex quadratic, x0 = -2:", [round(float(v), 6) for v in newton(f_convex, -2.0, 3)])


def f_exp(x):  # convex, not quadratic: minimum at x* = ln 2
    return jnp.exp(x) - 2 * x


p_newton = newton(f_exp, 2.0, 6)
p_gd = gradient_descent(jax.grad(f_exp), 2.0, lr=0.1, n_steps=6)
print("\n k   |x_k - ln2| Newton   |x_k - ln2| GD (lr=0.1)")
for k in range(7):
    print(f"{k:2d}   {abs(float(p_newton[k]) - np.log(2)):18.2e}   {abs(float(p_gd[k]) - np.log(2)):18.2e}")

**Newton on the rugged function.** Newton solves $f'(x)=0$: it cannot tell a minimum from a maximum.
Where $f''(x) < 0$ the parabola opens downwards and the step goes *uphill*.

In [ ]:
fig, ax = plt.subplots()
ax.plot(grid, f_rugged(grid), color="tab:blue", lw=1)
d2f_rugged = jax.grad(jax.grad(f_rugged))
for x0, color in [(-0.8, "tab:green"), (-0.5, "tab:red")]:
    p = newton(f_rugged, x0, 8)
    kind = "minimum" if float(d2f_rugged(p[-1])) > 0 else "MAXIMUM"
    ax.plot(p, f_rugged(p), "o-", color=color, ms=5, label=f"x0={x0}: converges to a {kind} at x={float(p[-1]):.3f}")
ax.set(xlim=(-2, 1), ylim=(-1.5, 1.2), xlabel="x", title="Newton finds stationary points, not necessarily minima")
ax.legend()
plt.show()
p = newton(f_rugged, -0.9, 8)
print(f"x0=-0.9: f''(x0) = {float(d2f_rugged(-0.9)):.2f}, the first step jumps to x = {float(p[1]):.2f}, "
      f"and Newton ends at x = {float(p[-1]):.2f}, far outside the plot")

**Newton in 2D: the Rosenbrock valley.** $\;f(x,y) = (1-x)^2 + 100\,(y-x^2)^2$, minimum at $(1,1)$.
`jax.hessian` gives the $2\times2$ matrix of second derivatives.

In [ ]:
def rosenbrock(p):
    x, y = p[..., 0], p[..., 1]
    return (1 - x) ** 2 + 100 * (y - x**2) ** 2


grad_rb = jax.jit(jax.grad(rosenbrock))
hess_rb = jax.jit(jax.hessian(rosenbrock))
p0 = jnp.array([-1.2, 1.0])
print("gradient at p0:", grad_rb(p0))
print("Hessian at p0:\n", hess_rb(p0))


def newton_nd(grad, hess, x0, n_steps):
    x = jnp.asarray(x0, dtype=float)
    path = [x]
    for _ in range(n_steps):
        x = x - jnp.linalg.solve(hess(x), grad(x))
        path.append(x)
    return jnp.stack(path)


path_newton = newton_nd(grad_rb, hess_rb, p0, 8)
path_gd = gradient_descent(grad_rb, p0, lr=1e-3, n_steps=5000)
print("Newton, 8 steps     :", path_newton[-1], " f =", float(rosenbrock(path_newton[-1])))
print("GD, 5000 steps      :", path_gd[-1], " f =", float(rosenbrock(path_gd[-1])))


def contour(ax, f, xlim=(-2, 2), ylim=(-1, 3), levels=None):
    X, Y = jnp.meshgrid(jnp.linspace(xlim[0], xlim[1], 300), jnp.linspace(ylim[0], ylim[1], 300))
    Z = f(jnp.stack([X, Y], axis=-1))
    ax.contour(X, Y, jnp.log1p(Z - Z.min()), levels=levels or 25, cmap="Greys", linewidths=0.7)
    ax.set(xlim=xlim, ylim=ylim, xlabel="x", ylabel="y")


fig, ax = plt.subplots(figsize=(7, 5))
contour(ax, rosenbrock)
ax.plot(*path_gd[::50].T, ".-", color="tab:blue", ms=3, label="GD, 5000 steps (every 50th)")
ax.plot(*path_newton.T, "o-", color="tab:red", ms=5, label="Newton, 8 steps")
ax.plot(1, 1, "k*", ms=14)
ax.set_title("Rosenbrock: curvature information pays off")
ax.legend()
plt.show()

**Takeaway:** Newton uses the curvature to pick both the direction and the step length.
Its cost ($n^2$ Hessian entries, an $n\times n$ solve) is why deep learning uses first-order methods, with cheap
per-parameter tricks that *imitate* curvature: momentum, RMSProp, Adam.

## G5 — Momentum, RMSProp and Adam

| method | update |
|:--|:--|
| GD | $\theta \leftarrow \theta - \eta\, g$ |
| Momentum | $v \leftarrow \beta v + g,\quad \theta \leftarrow \theta - \eta v$ |
| Nesterov | $g$ evaluated at the look-ahead point $\theta - \eta\beta v$ |
| RMSProp | $s \leftarrow \rho s + (1-\rho) g^2,\quad \theta \leftarrow \theta - \eta\, g / (\sqrt{s}+\epsilon)$ |
| Adam | $m \leftarrow \beta_1 m + (1-\beta_1) g,\ \ s \leftarrow \beta_2 s + (1-\beta_2) g^2,\ \ \theta \leftarrow \theta - \eta\, \hat m / (\sqrt{\hat s}+\epsilon)$ |

with $g = \nabla\mathcal{L}(\theta)$ and the bias corrections $\hat m = m/(1-\beta_1^t)$, $\hat s = s/(1-\beta_2^t)$.

In [ ]:
def momentum(grad, x0, lr, n_steps, beta=0.9, nesterov=False):
    x = jnp.asarray(x0, dtype=float)
    v = jnp.zeros_like(x)
    path = [x]
    for _ in range(n_steps):
        g = grad(x - lr * beta * v) if nesterov else grad(x)
        v = beta * v + g
        x = x - lr * v
        path.append(x)
    return jnp.stack(path)


def rmsprop(grad, x0, lr, n_steps, rho=0.9, eps=1e-8):
    x = jnp.asarray(x0, dtype=float)
    s = jnp.zeros_like(x)
    path = [x]
    for _ in range(n_steps):
        g = grad(x)
        s = rho * s + (1 - rho) * g**2
        x = x - lr * g / (jnp.sqrt(s) + eps)
        path.append(x)
    return jnp.stack(path)


def adam(grad, x0, lr, n_steps, beta1=0.9, beta2=0.999, eps=1e-8):
    x = jnp.asarray(x0, dtype=float)
    m = jnp.zeros_like(x)
    s = jnp.zeros_like(x)
    path = [x]
    for t in range(1, n_steps + 1):
        g = grad(x)
        m = beta1 * m + (1 - beta1) * g
        s = beta2 * s + (1 - beta2) * g**2
        m_hat, s_hat = m / (1 - beta1**t), s / (1 - beta2**t)
        x = x - lr * m_hat / (jnp.sqrt(s_hat) + eps)
        path.append(x)
    return jnp.stack(path)

**An ill-conditioned bowl.** $q(x,y) = \tfrac12(x^2 + 25y^2)$: 25 times steeper along $y$ than along $x$.
The largest stable learning rate is set by the steep direction, so plain GD crawls along the flat one and
zig-zags across the steep one.

In [ ]:
def bowl(p):
    return 0.5 * (p[..., 0] ** 2 + 25 * p[..., 1] ** 2)


grad_bowl = jax.jit(jax.grad(bowl))
q0 = jnp.array([-4.5, 1.5])
runs = {
    "GD (lr=0.075)": gradient_descent(grad_bowl, q0, 0.075, 100),
    "Momentum (lr=0.03, beta=0.7)": momentum(grad_bowl, q0, 0.03, 100, beta=0.7),
    "RMSProp (lr=0.1)": rmsprop(grad_bowl, q0, 0.1, 100),
    "Adam (lr=0.2)": adam(grad_bowl, q0, 0.2, 100),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
contour(axes[0], bowl, xlim=(-5, 5), ylim=(-2, 2))
for (name, p), color in zip(runs.items(), ["tab:blue", "tab:orange", "tab:green", "tab:red"]):
    axes[0].plot(*p.T, ".-", color=color, ms=3, lw=1, label=name)
    axes[1].semilogy(bowl(p), color=color, label=name)
axes[0].set_title("paths on q(x, y)")
axes[1].set(xlabel="iteration", ylabel="loss", title="loss (log scale)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

* GD is limited by the steep direction ($\eta < 2/25$) and zig-zags.
* Momentum averages the zig-zag out (opposite gradients cancel in $v$) and accumulates speed along the flat direction.
* RMSProp and Adam divide each coordinate by its own gradient size, so both directions move at a similar pace:
  they head almost straight to the minimum, then hover around it with a step of order $\eta$ (the learning rate is
  usually decayed for that reason).

**The Rosenbrock valley, 3000 steps each.** Each learning rate was tuned roughly by hand. Adam and RMSProp rescale
each coordinate by its recent gradient size, which is a cheap, diagonal stand-in for Newton's curvature.

In [ ]:
runs = {
    "GD (lr=1e-3)": gradient_descent(grad_rb, p0, 1e-3, 3000),
    "Momentum (lr=1e-3)": momentum(grad_rb, p0, 1e-3, 3000, beta=0.9),
    "Nesterov (lr=1e-3)": momentum(grad_rb, p0, 1e-3, 3000, beta=0.9, nesterov=True),
    "RMSProp (lr=3e-3)": rmsprop(grad_rb, p0, 3e-3, 3000),
    "Adam (lr=0.05)": adam(grad_rb, p0, 0.05, 3000),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
contour(axes[0], rosenbrock, xlim=(-1.6, 1.6), ylim=(-0.5, 1.8))
for (name, p), color in zip(runs.items(), ["tab:blue", "tab:orange", "tab:purple", "tab:green", "tab:red"]):
    axes[0].plot(*p[::20].T, ".-", color=color, ms=2, lw=1, label=name)
    axes[1].semilogy(rosenbrock(p) + 1e-16, color=color, label=f"{name}: f={float(rosenbrock(p[-1])):.1e}")
axes[0].plot(1, 1, "k*", ms=14)
axes[1].set(xlabel="iteration", ylabel="loss", title="Rosenbrock loss (log scale)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Momentum on the rugged 1D function.** A heavy ball keeps its velocity and can roll over small bumps that trap GD.

In [ ]:
fig, ax = plt.subplots()
ax.plot(grid, f_rugged(grid), color="tab:blue", lw=1)
x0 = -3.7
for name, p, color in [
    ("GD, lr=0.01", gradient_descent(df_rugged, x0, 0.01, 1000), "tab:red"),
    ("Momentum, lr=0.01, beta=0.95", momentum(df_rugged, x0, 0.01, 1000, beta=0.95), "tab:green"),
]:
    ax.plot(p, f_rugged(p), ".-", color=color, ms=2, lw=0.8, alpha=0.7)
    ax.plot(p[-1], f_rugged(p[-1]), "o", color=color, ms=10, label=f"{name}: ends at x={float(p[-1]):+.2f}")
ax.set(xlabel="x", ylabel="f(x)", title=f"Start at x0 = {x0}")
ax.legend()
plt.show()


def heavy_ball_final(x0, lr, beta, n_steps=1000):
    def body(_, state):
        x, v = state
        v = beta * v + jax.grad(f_rugged)(x)
        return x - lr * v, v

    return jax.lax.fori_loop(0, n_steps, body, (x0, 0.0))[0]


for beta in [0.0, 0.95]:
    finals = jax.vmap(lambda x0: heavy_ball_final(x0, 0.01, beta))(x0s)
    print(f"beta = {beta:4.2f}: over 400 starts, mean final f = {float(jnp.mean(f_rugged(finals))):+.2f}, "
          f"global basin reached from {float(jnp.mean(jnp.abs(finals + 0.67) < 0.05)):.0%}")

**Takeaway:** momentum lowers the *typical* final loss (it skips the shallowest ripples), but it is still a local
method: most starts do not reach the global minimum.

**Stochastic gradients (SGD).** In ML the loss is an average over $N$ examples,
$\mathcal{L}(\theta) = \frac1N\sum_i \ell(\theta; x_i, y_i)$. A **mini-batch** of $B \ll N$ examples gives a cheap,
*noisy* estimate of the gradient. Here: fit a line $\hat y = wx + b$ to 1000 points.

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-1, 1, 1000)
Y = 2.0 * X - 1.0 + rng.normal(0, 0.3, X.size)


def line_loss(theta, x, y):
    return jnp.mean((theta[0] * x + theta[1] - y) ** 2)


line_grad = jax.jit(jax.grad(line_loss))


def sgd(theta0, lr, n_steps, batch):
    theta = jnp.asarray(theta0, dtype=float)
    path = [theta]
    for _ in range(n_steps):
        idx = rng.integers(0, X.size, batch)
        theta = theta - lr * line_grad(theta, X[idx], Y[idx])
        path.append(theta)
    return jnp.stack(path)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
W, B = jnp.meshgrid(jnp.linspace(-1, 4, 200), jnp.linspace(-3, 2, 200))
L = jax.vmap(jax.vmap(lambda w, b: line_loss(jnp.array([w, b]), X, Y)))(W, B)
for ax in axes:
    ax.contour(W, B, L, levels=40, cmap="Greys", linewidths=0.7)
for batch, color in [(1000, "tab:blue"), (16, "tab:orange"), (1, "tab:red")]:
    p = sgd([-0.5, 1.5], lr=0.2, n_steps=200, batch=batch)
    for ax in axes:
        ax.plot(*p.T, ".-", color=color, ms=2, lw=1, label=f"batch = {batch}: w={float(p[-1, 0]):.2f}, b={float(p[-1, 1]):.2f}")
for ax in axes:
    ax.plot(2, -1, "k*", ms=14)
    ax.set(xlabel="w", ylabel="b")
axes[0].set_title("Full batch vs. mini-batch SGD (200 steps, lr = 0.2)")
axes[1].set(xlim=(1.5, 2.5), ylim=(-1.4, -0.6), title="zoom on the minimum")
axes[0].legend()
plt.tight_layout()
plt.show()

**Takeaway:** smaller batches are cheaper per step and noisier. The noise is not only a nuisance: it helps escape
sharp local minima and saddle points. Adam with mini-batches is the default optimizer of deep learning (Keras
`optimizer="adam"`).

## G6 — JAX for custom models and custom losses

JAX transforms ordinary Python + `jax.numpy` functions:

| transform | what it gives |
|:--|:--|
| `jax.grad(f)` | $\nabla f$ (reverse-mode autodiff, exact to machine precision) |
| `jax.value_and_grad(f)` | $f$ and $\nabla f$ in one pass |
| `jax.hessian(f)` | the matrix of second derivatives |
| `jax.jit(f)` | the function compiled with XLA |
| `jax.vmap(f)` | the function vectorised over a batch axis |

So a **new model** or a **new loss** needs no derivative by hand: write the forward computation, and JAX derives it.

In [ ]:
slow = jax.grad(rosenbrock)
fast = jax.jit(jax.grad(rosenbrock))
fast(p0).block_until_ready()  # the first call compiles
for name, fn in [("grad", slow), ("jit(grad)", fast)]:
    t0 = time.perf_counter()
    for _ in range(2000):
        fn(p0).block_until_ready()
    print(f"{name:10s}: {(time.perf_counter() - t0) / 2000 * 1e6:7.1f} us per call")

**A custom model with a custom loss.** A damped oscillator $\hat y(t) = A\,e^{-\lambda t}\cos(\omega t + \varphi)$,
fitted with the **Huber loss** (quadratic for small residuals, linear for large ones, so outliers pull less):

$$\ell_\delta(r) = \begin{cases} \tfrac12 r^2 & |r| \le \delta \\ \delta\,(|r| - \tfrac12\delta) & \text{otherwise} \end{cases}$$

In [ ]:
rng = np.random.default_rng(1)
t = jnp.linspace(0, 10, 200)
true = dict(A=2.0, lam=0.3, omega=2.0, phi=0.5)
y_clean = true["A"] * jnp.exp(-true["lam"] * t) * jnp.cos(true["omega"] * t + true["phi"])
y_obs = y_clean + rng.normal(0, 0.1, t.size)
outliers = rng.choice(t.size, 12, replace=False)
y_obs = y_obs.at[outliers].add(rng.choice([-1.5, 1.5], 12))


def oscillator(params, t):
    A, lam, omega, phi = params
    return A * jnp.exp(-lam * t) * jnp.cos(omega * t + phi)


def huber(r, delta=0.2):
    return jnp.where(jnp.abs(r) <= delta, 0.5 * r**2, delta * (jnp.abs(r) - 0.5 * delta))


def loss_huber(params):
    return jnp.mean(huber(oscillator(params, t) - y_obs))


def loss_mse(params):
    return jnp.mean((oscillator(params, t) - y_obs) ** 2)


def fit(loss, params0, lr=0.02, n_steps=3000):
    step = jax.jit(jax.value_and_grad(loss))
    params = jnp.asarray(params0, dtype=float)
    m, s = jnp.zeros_like(params), jnp.zeros_like(params)
    for k in range(1, n_steps + 1):  # Adam, written out as in G5
        _, g = step(params)
        m, s = 0.9 * m + 0.1 * g, 0.999 * s + 0.001 * g**2
        params = params - lr * (m / (1 - 0.9**k)) / (jnp.sqrt(s / (1 - 0.999**k)) + 1e-8)
    return params


init = [1.0, 0.1, 1.8, 0.0]
fits = {"MSE": fit(loss_mse, init), "Huber": fit(loss_huber, init)}
print(f"{'':6s}{'A':>8s}{'lambda':>8s}{'omega':>8s}{'phi':>8s}")
print(f"{'true':6s}" + "".join(f"{v:8.3f}" for v in true.values()))
for name, p in fits.items():
    print(f"{name:6s}" + "".join(f"{float(v):8.3f}" for v in p))

fig, ax = plt.subplots()
ax.plot(t, y_obs, "k.", ms=4, label="data (12 outliers)")
ax.plot(t, y_clean, "g--", label="true signal")
for (name, p), color in zip(fits.items(), ["tab:orange", "tab:blue"]):
    ax.plot(t, oscillator(p, t), color=color, lw=2, label=f"fit with {name}")
ax.set(xlabel="t", ylabel="y", title="Same model, two losses: JAX derives both gradients")
ax.legend()
plt.show()

**Differentiating through a numerical integration.** Some models have no formula, only a simulation. Logistic
growth $\frac{dy}{dt} = r\,y\,(1 - y/K)$ is integrated with Euler steps inside `jax.lax.scan`, and JAX
differentiates the *whole simulation* with respect to $r$ and $K$.

In [ ]:
dt, n_t = 0.1, 100
t_ode = jnp.arange(n_t) * dt


def simulate(params, y0=0.1):
    r, K = params

    def euler(y, _):
        y_next = y + dt * r * y * (1 - y / K)
        return y_next, y

    _, ys = jax.lax.scan(euler, y0, None, length=n_t)
    return ys


rng = np.random.default_rng(2)
y_data = simulate(jnp.array([0.8, 5.0])) + rng.normal(0, 0.15, n_t)


def ode_loss(params):
    return jnp.mean((simulate(params) - y_data) ** 2)


print("d loss / d(r, K) at (0.5, 3.0):", jax.grad(ode_loss)(jnp.array([0.5, 3.0])))
params = fit(ode_loss, [0.5, 3.0], lr=0.05, n_steps=1500)
print(f"fitted r = {float(params[0]):.3f} (true 0.8),  K = {float(params[1]):.3f} (true 5.0)")

fig, ax = plt.subplots()
ax.plot(t_ode, y_data, "k.", ms=4, label="noisy observations")
ax.plot(t_ode, simulate(jnp.array([0.5, 3.0])), "--", color="tab:gray", label="initial guess r=0.5, K=3")
ax.plot(t_ode, simulate(params), color="tab:blue", lw=2, label="fitted through the ODE solver")
ax.set(xlabel="t", ylabel="y", title="Gradients through 100 Euler steps")
ax.legend()
plt.show()

**Takeaway:** with JAX the workflow for *any* differentiable model is the same three lines:
write the model, write the loss, call `jax.grad`. This is what Keras does internally (on the JAX backend) when it
trains a network, and what we will use whenever a model or loss is not in a library.